In [1]:
import pandas as pd
import polars as pl

In [2]:
def load_dataset(url:str) -> pd.DataFrame:
    return pd.read_parquet(url)

In [3]:
# df_devolucoes = load_dataset("https://bkt-panvel-puc-projcd.s3.amazonaws.com/project-puc/devolucoes.parquet")
# df_filiais = load_dataset("https://bkt-panvel-puc-projcd.s3.amazonaws.com/project-puc/filiais.parquet")
# df_metas = load_dataset("https://bkt-panvel-puc-projcd.s3.amazonaws.com/project-puc/metas.parquet")
# df_vendas = load_dataset("https://bkt-panvel-puc-projcd.s3.amazonaws.com/project-puc/vendas.parquet")

# df_devolucoes.to_parquet("../../data/devolucoes.parquet")
# df_filiais.to_parquet("../../data/filiais.parquet")
# df_metas.to_parquet("../../data/metas.parquet")
# df_vendas.to_parquet("../../data/vendas.parquet")

df_devolucoes = pl.read_parquet("../../../data/devolucoes.parquet")
df_filiais = pl.read_parquet("../../../data/filiais.parquet")
df_metas = pl.read_parquet("../../../data/metas.parquet")
df_vendas = pl.read_parquet("../../../data/vendas.parquet")

In [4]:
datasets = {
    "devolucoes": df_devolucoes,
    "filiais": df_filiais,
    "metas": df_metas,
    "vendas": df_vendas}

for name, df in datasets.items():
    print(f"Dataset: {name}")
    print(df.columns)
    print("\n")

Dataset: devolucoes
['codigo_filial', 'data_devolucao', 'categoria_gerencial', 'quantidade', 'valor_devolucao']


Dataset: filiais
['codigo_filial', 'faixa_vida', 'localidade', 'uf', 'tipo_estabelecimento', 'delivery', 'metragem_area_venda', 'panvel_clinic', 'estacionamento', 'atendimento_24_horas']


Dataset: metas
['codigo_filial', 'data_meta_venda', 'meta_n_med', 'meta_med', 'valor_meta_venda']


Dataset: vendas
['codigo_documento_saida', 'codigo_filial', 'data_emissao', 'categoria_gerencial', 'quantidade', 'faturamento']




In [5]:
df_devolucoes.describe()

statistic,codigo_filial,data_devolucao,categoria_gerencial,quantidade,valor_devolucao
str,f64,str,str,f64,f64
"""count""",85677.0,"""85677""","""85677""",85677.0,85677.0
"""null_count""",0.0,"""0""","""0""",0.0,0.0
"""mean""",1669.348787,"""2025-01-16 20:02:48.535000+00:…",null,13.641759,790.317216
"""std""",101.015254,null,null,207.48101,5191.981902
"""min""",1500.0,"""2024-01-01 03:00:00+00:00""","""MED""",3.0,0.03
"""25%""",1578.0,"""2024-07-20 03:00:00+00:00""",null,3.0,125.04
"""50%""",1668.0,"""2025-01-26 03:00:00+00:00""",null,6.0,281.72
"""75%""",1755.0,"""2025-07-20 03:00:00+00:00""",null,12.0,604.27
"""max""",1887.0,"""2025-12-31 03:00:00+00:00""","""N-MED""",49419.0,541056.1


In [6]:
df_filiais.describe()

statistic,codigo_filial,faixa_vida,localidade,uf,tipo_estabelecimento,delivery,metragem_area_venda,panvel_clinic,estacionamento,atendimento_24_horas
str,f64,str,str,str,str,str,f64,str,str,str
"""count""",124.0,"""124""","""124""","""124""","""124""","""124""",124.0,"""124""","""124""","""124"""
"""null_count""",0.0,"""0""","""0""","""0""","""0""","""0""",0.0,"""0""","""0""","""0"""
"""mean""",1696.790323,null,null,null,null,null,510.703274,null,null,null
"""std""",114.020486,null,null,null,null,null,101.204792,null,null,null
"""min""",1500.0,"""ENTRE 1-2 ANOS""","""APUCARANA""","""PR""","""BAIRRO""","""NÃO""",263.76,"""NÃO""","""NÃO""","""NÃO"""
"""25%""",1599.0,null,null,null,null,null,471.0,null,null,null
"""50%""",1701.0,null,null,null,null,null,512.5422,null,null,null
"""75%""",1794.0,null,null,null,null,null,549.5,null,null,null
"""max""",1887.0,"""MENOS DE 1 ANO""","""UMUARAMA""","""PR""","""SUPERMERCADO""","""SIM""",1140.7306,"""SIM""","""SIM""","""SIM"""


In [7]:
# Calcular a média geral de cada meta
medias_metas = df_metas.select([
    pl.col("meta_med").mean().alias("Média Meta MED"),
    pl.col("meta_n_med").mean().alias("Média Meta N-MED")
])
print(medias_metas)

# Contar em quantos registros a meta MED é maior que a N-MED
contagem_comparativa = df_metas.select([
    (pl.col("meta_med") > pl.col("meta_n_med")).sum().alias("MED > N-MED"),
    (pl.col("meta_n_med") > pl.col("meta_med")).sum().alias("N-MED > MED")
])
print("\nComparação de frequência:")
print(contagem_comparativa)

shape: (1, 2)
┌────────────────┬──────────────────┐
│ Média Meta MED ┆ Média Meta N-MED │
│ ---            ┆ ---              │
│ f64            ┆ f64              │
╞════════════════╪══════════════════╡
│ 46167.3284     ┆ 27238.357137     │
└────────────────┴──────────────────┘

Comparação de frequência:
shape: (1, 2)
┌─────────────┬─────────────┐
│ MED > N-MED ┆ N-MED > MED │
│ ---         ┆ ---         │
│ u32         ┆ u32         │
╞═════════════╪═════════════╡
│ 77547       ┆ 2230        │
└─────────────┴─────────────┘


In [8]:
# Exemplo de preparação de dados no Polars
df_plot_areas = df_metas.with_columns(
    pl.col("data_meta_venda").cast(pl.Date).dt.truncate("1mo").alias("mes_ano")
).group_by("mes_ano").agg([
    pl.col("meta_med").sum(),
    pl.col("meta_n_med").sum()
]).sort("mes_ano")

In [9]:
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Configuração global de estilo para padronizar o relatório
sns.set_theme(style="whitegrid")
cor_azul_escuro = '#003366'
cor_azul_claro = '#4A90E2'
cor_laranja = '#FF9F00'

# ==========================================
# GRÁFICO 1: Distribuição da Faixa de Vida
# ==========================================
# Utilizando os dados já mapeados
df_faixa = pl.DataFrame({
    "faixa_vida": ["MENOS DE 1 ANO", "ENTRE 1-2 ANOS", "ENTRE 2-3 ANOS", "MAIS DE 3 ANOS"],
    "quantidade": [11, 12, 7, 94]
})

plt.figure(figsize=(7, 4.5))
ax1 = sns.barplot(x="faixa_vida", y="quantidade", data=df_faixa.to_pandas(), palette="Blues_d")

plt.title("Distribuição das Filiais por Tempo de Operação", fontsize=12, weight='bold', color=cor_azul_escuro)
plt.xlabel("")
plt.ylabel("Número de Filiais", fontsize=10)
plt.xticks(fontsize=9)

# Adicionando rótulos no topo
for p in ax1.patches:
    ax1.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height()),
                 ha='center', va='center', xytext=(0, 6), textcoords='offset points', fontsize=10, weight='bold')

plt.tight_layout()
plt.savefig("grafico1_faixa_vida.png", dpi=300)
plt.close()


# ==========================================
# GRÁFICO 2: Rosca (Donut) de Dominância
# ==========================================
# Supondo que você já carregou seu df_metas. 
# Exemplo de cálculo usando Polars:
med_maior = df_metas.filter(pl.col("meta_med") > pl.col("meta_n_med")).height
nmed_maior = df_metas.filter(pl.col("meta_n_med") >= pl.col("meta_med")).height

# Dados para o gráfico
tamanhos = [med_maior, nmed_maior]
labels = ['MED > N-MED', 'N-MED ≥ MED']

plt.figure(figsize=(5, 5))
plt.pie(tamanhos, labels=labels, autopct='%1.1f%%', startangle=90, 
        colors=[cor_azul_claro, cor_laranja], pctdistance=0.75, 
        textprops={'fontsize': 10, 'weight': 'bold'})

# Transformando a pizza em rosca
circulo_centro = plt.Circle((0,0), 0.60, fc='white')
fig = plt.gcf()
fig.gca().add_artist(circulo_centro)

plt.title("Dominância de Valor nas Metas Diárias", fontsize=12, weight='bold', color=cor_azul_escuro)
plt.tight_layout()
plt.savefig("grafico2_rosca_dominancia.png", dpi=300)
plt.close()


# ==========================================
# GRÁFICO 3: Média e Mediana (Barras Agrupadas)
# ==========================================
# Calculando Média e Mediana usando Polars
estatisticas = df_metas.select([
    pl.col("meta_med").mean().alias("MED_Media"),
    pl.col("meta_med").median().alias("MED_Mediana"),
    pl.col("meta_n_med").mean().alias("NMED_Media"),
    pl.col("meta_n_med").median().alias("NMED_Mediana")
]).to_pandas().iloc[0]

# Organizando os dados para o Seaborn (formato longo)
df_plot_stats = pl.DataFrame({
    "Métrica": ["Média", "Média", "Mediana", "Mediana"],
    "Categoria": ["MED", "N-MED", "MED", "N-MED"],
    "Valor": [estatisticas["MED_Media"], estatisticas["NMED_Media"], 
              estatisticas["MED_Mediana"], estatisticas["NMED_Mediana"]]
}).to_pandas()

plt.figure(figsize=(7, 4.5))
ax3 = sns.barplot(x="Métrica", y="Valor", hue="Categoria", data=df_plot_stats, palette=[cor_azul_claro, cor_laranja])

plt.title("Comparativo: Média vs Mediana das Metas", fontsize=12, weight='bold', color=cor_azul_escuro)
plt.xlabel("")
plt.ylabel("Valor (R$)", fontsize=10)
plt.legend(title="Categoria")

# Adicionando rótulos de valores nas barras
for p in ax3.patches:
    ax3.annotate(f'R$ {p.get_height():,.0f}', (p.get_x() + p.get_width() / 2., p.get_height()),
                 ha='center', va='center', xytext=(0, 6), textcoords='offset points', fontsize=9, weight='bold')

plt.tight_layout()
plt.savefig("grafico3_media_mediana.png", dpi=300)
plt.close()

print("Imagens geradas com sucesso: grafico1_faixa_vida.png, grafico2_rosca_dominancia.png, grafico3_media_mediana.png")

C:\Users\arthu\AppData\Local\Temp\ipykernel_20840\4123719925.py:22: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax1 = sns.barplot(x="faixa_vida", y="quantidade", data=df_faixa.to_pandas(), palette="Blues_d")


Imagens geradas com sucesso: grafico1_faixa_vida.png, grafico2_rosca_dominancia.png, grafico3_media_mediana.png


In [10]:
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.lines import Line2D

# Configuração de estilo e cores
sns.set_theme(style="whitegrid")
cor_azul_escuro = '#003366'
cor_azul_claro = '#4A90E2'
cor_laranja = '#FF9F00'

# ==========================================
# GRÁFICO 3: Boxplot com Média (Tracejada) e Mediana
# ==========================================

# 1. Preparação dos dados (convertendo para formato longo)
df_plot_box = df_metas.select(["meta_med", "meta_n_med"]).to_pandas().melt(var_name="Categoria", value_name="Valor")
df_plot_box["Categoria"] = df_plot_box["Categoria"].replace({"meta_med": "MED", "meta_n_med": "N-MED"})

plt.figure(figsize=(7, 4.5))

# 2. Gerando o Boxplot
ax3 = sns.boxplot(
    x="Categoria", 
    y="Valor", 
    data=df_plot_box, 
    palette=[cor_azul_claro, cor_laranja],
    showfliers=False,  # Oculta os outliers
    showmeans=True,    # Mostra a média
    meanline=True,     # Transforma o marcador da média em uma linha
    meanprops={        # Estiliza a linha da média
        "color": "red", 
        "linestyle": "--", 
        "linewidth": 2
    } 
)

plt.title("Distribuição das Metas Diárias (Mediana e Média)", fontsize=12, weight='bold', color=cor_azul_escuro)
plt.xlabel("")
plt.ylabel("Valor (R$)", fontsize=10)

# 3. Atualizando a legenda para refletir as linhas
legend_elements = [
    Line2D([0], [0], color='black', lw=2, label='Mediana (Linha Sólida)'),
    Line2D([0], [0], color='red', lw=2, linestyle='--', label='Média (Linha Tracejada)')
]
plt.legend(handles=legend_elements, loc='upper right', fontsize=9)

plt.tight_layout()
plt.savefig("grafico3_boxplot_metas.png", dpi=300)
plt.close()

print("Imagem atualizada com sucesso: grafico3_boxplot_metas.png")

C:\Users\arthu\AppData\Local\Temp\ipykernel_20840\862434148.py:23: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax3 = sns.boxplot(


Imagem atualizada com sucesso: grafico3_boxplot_metas.png


In [11]:
df_filiais["tipo_estabelecimento"].value_counts()

tipo_estabelecimento,count
str,u32
"""SUPERMERCADO""",1
"""MALL""",3
"""BAIRRO""",76
"""SHOPPING""",6
"""CENTRO""",38


In [12]:
df_metas.describe()

statistic,codigo_filial,data_meta_venda,meta_n_med,meta_med,valor_meta_venda
str,f64,str,f64,f64,f64
"""count""",81268.0,"""81268""",81268.0,81268.0,81268.0
"""null_count""",0.0,"""0""",0.0,0.0,0.0
"""mean""",1676.587919,"""2025-01-13 13:58:27.683000+00:…",27238.357137,46167.3284,73405.685537
"""std""",102.929091,null,17270.506971,29280.783487,45677.234169
"""min""",1500.0,"""2024-01-01 03:00:00+00:00""",0.0,-11729.44,0.0
"""25%""",1587.0,"""2024-07-17 03:00:00+00:00""",17972.48,29698.53,48442.35
"""50%""",1677.0,"""2025-01-21 03:00:00+00:00""",24122.83,41372.83,65704.4
"""75%""",1764.0,"""2025-07-16 03:00:00+00:00""",32235.81,56045.99,87833.65
"""max""",1887.0,"""2025-12-31 03:00:00+00:00""",413956.34,510918.63,918984.93


In [13]:
df_vendas.describe()

statistic,codigo_documento_saida,codigo_filial,data_emissao,categoria_gerencial,quantidade,faturamento
str,f64,f64,str,str,f64,f64
"""count""",2.0863735e7,2.0863735e7,"""20863735""","""20863735""",2.0863735e7,2.0863735e7
"""null_count""",0.0,0.0,"""0""","""0""",0.0,0.0
"""mean""",663272.660371,1661.35975,"""2025-01-18 19:02:23.642000+00:…",null,7.353185,284.964194
"""std""",628784.590016,99.918801,null,null,32.882729,1119.75891
"""min""",1.0,1500.0,"""2024-01-01 00:00:00+00:00""","""MED""",3.0,0.03
"""25%""",229204.0,1566.0,"""2024-07-23 00:00:00+00:00""",null,3.0,56.51
"""50%""",531266.0,1659.0,"""2025-01-27 00:00:00+00:00""",null,6.0,127.34
"""75%""",857004.0,1746.0,"""2025-07-23 00:00:00+00:00""",null,9.0,301.0
"""max""",3.677479e6,1887.0,"""2025-12-31 00:00:00+00:00""","""N-MED""",69120.0,775538.53


In [14]:
# Contar dias únicos e ordenar do menor para o maior
lacunas = df_vendas.group_by("codigo_filial").agg(pl.col("data_emissao").n_unique().alias("dias_com_venda")
).sort("dias_com_venda", descending=True)

print(lacunas.head(10))

shape: (10, 2)
┌───────────────┬────────────────┐
│ codigo_filial ┆ dias_com_venda │
│ ---           ┆ ---            │
│ decimal[4,0]  ┆ u32            │
╞═══════════════╪════════════════╡
│ 1587          ┆ 731            │
│ 1602          ┆ 731            │
│ 1596          ┆ 731            │
│ 1581          ┆ 731            │
│ 1575          ┆ 731            │
│ 1593          ┆ 731            │
│ 1605          ┆ 731            │
│ 1614          ┆ 731            │
│ 1638          ┆ 731            │
│ 1608          ┆ 731            │
└───────────────┴────────────────┘


In [15]:
import polars as pl
from datetime import date

# 1. Criar o intervalo de datas esperado (01/01/2024 a 31/12/2025)
data_inicial = date(2024, 1, 1)
data_final = date(2025, 12, 31)

calendario_esperado = pl.date_range(data_inicial, data_final, "1d", eager=True).alias("data_emissao")
calendario_df = pl.DataFrame(calendario_esperado)

# 2. Obter as datas que realmente possuem vendas (certifique-se de que a coluna é Date)
datas_venda_real = df_vendas.select(pl.col("data_emissao").cast(pl.Date)).unique()

# 3. Encontrar os dias que estão no calendário mas não nas vendas (Anti-join)
dias_sem_venda = calendario_df.join(datas_venda_real, on="data_emissao", how="anti")

print(f"Total de dias sem nenhuma venda na rede: {dias_sem_venda.height}")
print(dias_sem_venda)

Total de dias sem nenhuma venda na rede: 0
shape: (0, 1)
┌──────────────┐
│ data_emissao │
│ ---          │
│ date         │
╞══════════════╡
└──────────────┘


In [16]:
# Realiza o join e conta o total de registros órfãos
total_vendas_sem_meta = df_vendas.select(["codigo_filial", "data_emissao"]).unique().join(
    df_metas.select(["codigo_filial", "data_meta_venda"]),
    left_on=["codigo_filial", "data_emissao"],
    right_on=["codigo_filial", "data_meta_venda"],
    how="anti"
).height

print(f"Total de ocorrências (Filial-Dia) de vendas sem meta: {total_vendas_sem_meta}")

Total de ocorrências (Filial-Dia) de vendas sem meta: 79965


In [17]:
total_meta_zero = df_metas.filter(pl.col("valor_meta_venda") == 0).height
meta_med_zero = df_metas.filter(pl.col("meta_med") == 0).height
meta_n_med_zero = df_metas.filter(pl.col("meta_n_med") == 0).height
ambas_zero = df_metas.filter((pl.col("meta_med") == 0) & (pl.col("meta_n_med") == 0)).height
print(f"Registros onde MED e N_MED são zero simultaneamente: {ambas_zero}")
print(f"Registros com meta total zero: {total_meta_zero}")
print(f"Registros com meta MED zero: {meta_med_zero}")
print(f"Registros com meta N_MED zero: {meta_n_med_zero}")

Registros onde MED e N_MED são zero simultaneamente: 1491
Registros com meta total zero: 1491
Registros com meta MED zero: 1491
Registros com meta N_MED zero: 1537


In [18]:
# 1. Contar dias únicos de venda por filial
dias_por_filial = df_vendas.group_by("codigo_filial").agg(
    pl.col("data_emissao").n_unique().alias("total_dias")
)

# 2. Filtrar apenas as que possuem exatamente 731 dias
filiais_completas = dias_por_filial.filter(pl.col("total_dias") == 731)

# 3. Obter o número total de filiais
total_filiais_completas = filiais_completas.height

print(f"Número de filiais com o histórico completo (731 dias): {total_filiais_completas}")

Número de filiais com o histórico completo (731 dias): 74


In [19]:
# Calcular a média geral de cada meta
medias_metas = df_metas.select([
    pl.col("meta_med").mean().alias("Média Meta MED"),
    pl.col("meta_n_med").mean().alias("Média Meta N-MED")
])
print(medias_metas)

# Contar em quantos registros a meta MED é maior que a N-MED
contagem_comparativa = df_metas.select([
    (pl.col("meta_med") > pl.col("meta_n_med")).sum().alias("MED > N-MED"),
    (pl.col("meta_n_med") > pl.col("meta_med")).sum().alias("N-MED > MED")
])
print("\nComparação de frequência:")
print(contagem_comparativa)

shape: (1, 2)
┌────────────────┬──────────────────┐
│ Média Meta MED ┆ Média Meta N-MED │
│ ---            ┆ ---              │
│ f64            ┆ f64              │
╞════════════════╪══════════════════╡
│ 46167.3284     ┆ 27238.357137     │
└────────────────┴──────────────────┘

Comparação de frequência:
shape: (1, 2)
┌─────────────┬─────────────┐
│ MED > N-MED ┆ N-MED > MED │
│ ---         ┆ ---         │
│ u32         ┆ u32         │
╞═════════════╪═════════════╡
│ 77547       ┆ 2230        │
└─────────────┴─────────────┘


In [20]:
df_devolucoes.sort("valor_devolucao", descending=True).head(10)

codigo_filial,data_devolucao,categoria_gerencial,quantidade,valor_devolucao
"decimal[4,0]","datetime[ms, UTC]",str,"decimal[9,4]",f64
1605,2024-03-11 03:00:00 UTC,"""N-MED""",6720.0000,541056.1
1758,2024-09-30 03:00:00 UTC,"""N-MED""",7113.0000,535518.48
1506,2024-04-15 03:00:00 UTC,"""MED""",54.0000,425740.34
1884,2025-12-11 03:00:00 UTC,"""N-MED""",8934.0000,359918.9
1506,2025-05-06 03:00:00 UTC,"""MED""",21.0000,322698.42
1884,2025-12-08 03:00:00 UTC,"""N-MED""",49419.0000,241019.39
1833,2024-12-04 03:00:00 UTC,"""N-MED""",6000.0000,231805.8
1521,2024-04-15 03:00:00 UTC,"""MED""",45.0000,228281.1
1671,2024-12-18 03:00:00 UTC,"""N-MED""",3156.0000,225506.46


In [21]:
import polars as pl

df_vendas.filter(pl.col("data_emissao") == pl.lit("2025-12-11 03:00:00").str.to_datetime().dt.replace_time_zone("UTC")).head()

codigo_documento_saida,codigo_filial,data_emissao,categoria_gerencial,quantidade,faturamento
"decimal[7,0]","decimal[4,0]","datetime[ms, UTC]",str,"decimal[9,4]",f64


In [22]:
df_metas.describe()

statistic,codigo_filial,data_meta_venda,meta_n_med,meta_med,valor_meta_venda
str,f64,str,f64,f64,f64
"""count""",81268.0,"""81268""",81268.0,81268.0,81268.0
"""null_count""",0.0,"""0""",0.0,0.0,0.0
"""mean""",1676.587919,"""2025-01-13 13:58:27.683000+00:…",27238.357137,46167.3284,73405.685537
"""std""",102.929091,null,17270.506971,29280.783487,45677.234169
"""min""",1500.0,"""2024-01-01 03:00:00+00:00""",0.0,-11729.44,0.0
"""25%""",1587.0,"""2024-07-17 03:00:00+00:00""",17972.48,29698.53,48442.35
"""50%""",1677.0,"""2025-01-21 03:00:00+00:00""",24122.83,41372.83,65704.4
"""75%""",1764.0,"""2025-07-16 03:00:00+00:00""",32235.81,56045.99,87833.65
"""max""",1887.0,"""2025-12-31 03:00:00+00:00""",413956.34,510918.63,918984.93


In [23]:
df_filiais.filter(df_filiais["codigo_filial"]=="1521").head()

codigo_filial,faixa_vida,localidade,uf,tipo_estabelecimento,delivery,metragem_area_venda,panvel_clinic,estacionamento,atendimento_24_horas
"decimal[4,0]",str,str,str,str,str,f64,str,str,str
1521,"""MAIS DE 3 ANOS""","""CURITIBA""","""PR""","""BAIRRO""","""SIM""",1140.7306,"""NÃO""","""SIM""","""NÃO"""


In [24]:
df_vendas.sort("faturamento", descending=True).head(10)

codigo_documento_saida,codigo_filial,data_emissao,categoria_gerencial,quantidade,faturamento
"decimal[7,0]","decimal[4,0]","datetime[ms, UTC]",str,"decimal[9,4]",f64
3484372,1521,2025-02-04 00:00:00 UTC,"""MED""",9.0000,775538.53
3348522,1521,2024-05-24 00:00:00 UTC,"""MED""",9.0000,775538.53
3411646,1521,2024-09-18 00:00:00 UTC,"""MED""",9.0000,775538.53
3347510,1521,2024-05-22 00:00:00 UTC,"""MED""",18.0000,668258.71
39385,1848,2025-09-30 00:00:00 UTC,"""N-MED""",7500.0000,627414.75
18594,1848,2025-05-09 00:00:00 UTC,"""N-MED""",7440.0000,591236.71
5664,1833,2024-12-04 00:00:00 UTC,"""N-MED""",15000.0000,579514.5
1806248,1548,2025-05-30 00:00:00 UTC,"""N-MED""",14400.0000,571410.72
3461067,1521,2024-12-17 00:00:00 UTC,"""MED""",18.0000,528788.42


In [25]:
df_vendas = df_vendas.with_columns(
    pl.col("data_emissao").dt.date()
)

df_metas = df_metas.with_columns(
    pl.col("data_meta_venda").dt.date()
)

df_devolucoes = df_devolucoes.with_columns(
    pl.col("data_devolucao").dt.date()
)

In [26]:
df_grouped = (
    df_vendas
    .group_by(["codigo_filial", "data_emissao", "categoria_gerencial"])
    .agg([
        pl.col("quantidade").sum().alias("total_quantidade"),
        pl.col("faturamento").sum().alias("total_faturamento")
    ])
)

print(df_grouped.head())

shape: (5, 5)
┌───────────────┬──────────────┬─────────────────────┬──────────────────┬───────────────────┐
│ codigo_filial ┆ data_emissao ┆ categoria_gerencial ┆ total_quantidade ┆ total_faturamento │
│ ---           ┆ ---          ┆ ---                 ┆ ---              ┆ ---               │
│ decimal[4,0]  ┆ date         ┆ str                 ┆ decimal[9,4]     ┆ f64               │
╞═══════════════╪══════════════╪═════════════════════╪══════════════════╪═══════════════════╡
│ 1668          ┆ 2024-03-16   ┆ N-MED               ┆ 804.0000         ┆ 18379.93          │
│ 1836          ┆ 2025-10-07   ┆ MED                 ┆ 888.0000         ┆ 33164.79          │
│ 1674          ┆ 2024-07-04   ┆ N-MED               ┆ 1167.0000        ┆ 26600.84          │
│ 1614          ┆ 2024-06-07   ┆ N-MED               ┆ 1656.0000        ┆ 52067.93          │
│ 1698          ┆ 2025-12-14   ┆ MED                 ┆ 1683.0000        ┆ 115306.43         │
└───────────────┴──────────────┴──────────────

In [27]:
# df_final = df_grouped.join(
#     df_metas,
#     left_on=["codigo_filial", "data_emissao"],
#     right_on=["codigo_filial", "data_meta_venda"],
#     how="left"
# )

# df_final.head()

In [ ]:
# df_final = (
#     df_grouped.join(df_devolucoes, left_on=["codigo_filial", "data_emissao","categoria_gerencial"], right_on=["codigo_filial", "data_devolucao","categoria_gerencial"], how="left")
#     .with_columns(
#         pl.col("total_faturamento") - pl.col("valor_devolucao"),
#         pl.col("total_quantidade") - pl.col("quantidade")
#     )
# )

In [ ]:
# df_final.head()

codigo_filial,data_emissao,categoria_gerencial,total_quantidade,total_faturamento,quantidade,valor_devolucao
"decimal[4,0]",date,str,"decimal[38,4]",f64,"decimal[9,4]",f64
1668,2024-03-16,"""N-MED""",null,null,null,null
1836,2025-10-07,"""MED""",null,null,null,null
1674,2024-07-04,"""N-MED""",1155.0000,26020.16,12.0000,580.68
1614,2024-06-07,"""N-MED""",1644.0000,51659.57,12.0000,408.36
1698,2025-12-14,"""MED""",1677.0000,113567.7,6.0000,1738.73


In [34]:
import polars as pl
import calendar
import holidays
from datetime import date

# 1. Função para calcular a Black Friday (4ª sexta-feira de Novembro)
def calcular_black_friday(anos):
    datas_bf = []
    for ano in anos:
        mes_calendario = calendar.monthcalendar(ano, 11)
        sextas = [semana[4] for semana in mes_calendario if semana[4] != 0]
        dia_bf = sextas[3]
        datas_bf.append(date(ano, 11, dia_bf))
    return pl.DataFrame({"data_emissao": datas_bf, "black_friday": 1})

# 2. Configurações Iniciais
anos = [2024, 2025, 2026]
subdivisao = 'PR' # Feriados Estaduais do Paraná

# 3. Gerar Feriados Oficiais (BR + PR)
feriados_br_pr = holidays.country_holidays('BR', subdiv=subdivisao, years=anos)
df_feriados = pl.DataFrame({
    "data_emissao": list(feriados_br_pr.keys()),
    "nome_feriado": list(feriados_br_pr.values()),
    "feriado": 1
}).with_columns(pl.col("data_emissao").cast(pl.Date))

# 4. Datas Comemorativas do GitHub
datas_alvo = [
    "Dia Internacional da Mulher", "Dia das Mães", "Dia Mundial da Criança",
    "Dia dos Namorados", "Dia dos Pais", "Dia do Idoso", "Dia do Professor",
    "Dia de Finados", "Véspera de Natal", "Réveillon"
]

dfs_comemorativas = []
for ano in anos:
    url = f"https://raw.githubusercontent.com/joaopbini/feriados-brasil/master/dados/comemorativas/csv/{ano}.csv"
    try:
        df_ano = pl.read_csv(url, has_header=False, new_columns=["data", "nome_data", "tipo"])
        dfs_comemorativas.append(df_ano)
    except Exception as e:
        print(f"Aviso: Não foi possível baixar {ano} - {e}")

df_comemorativas = pl.concat(dfs_comemorativas).with_columns(
    pl.col("data").str.to_date("%d/%m/%Y").alias("data_emissao")
).filter(
    pl.col("nome_data").is_in(datas_alvo)
).with_columns(
    pl.lit(1).alias("data_comercial")
).select(["data_emissao", "data_comercial", "nome_data"])

df_bf = calcular_black_friday(anos)

df_final = df_grouped.join(
    df_feriados, on="data_emissao", how="left"
).join(
    df_comemorativas, on="data_emissao", how="left"
).join(
    df_bf, on="data_emissao", how="left"
)

df_final = df_final.with_columns(
    pl.col("feriado").fill_null(0),
    pl.col("nome_feriado").fill_null("Dia Comum"),
    pl.col("data_comercial").fill_null(0),
    pl.col("black_friday").fill_null(0),
    pl.col("nome_data").fill_null("Nenhuma")
)

In [40]:
df_amostra_datas = df_final.filter(
    (pl.col("black_friday") == 1) | 
    (pl.col("nome_data").is_in(["Dia das Mães", "Dia dos Namorados"])) |
    (pl.col("nome_feriado") == "Sexta-feira Santa")
)

df_amostra_datas.head()

codigo_filial,data_emissao,categoria_gerencial,total_quantidade,total_faturamento,nome_feriado,feriado,data_comercial,nome_data,black_friday
"decimal[4,0]",date,str,"decimal[9,4]",f64,str,i32,i32,str,i32
1602,2024-06-12,"""N-MED""",804.0000,19932.52,"""Dia Comum""",0,1,"""Dia dos Namorados""",0
1581,2024-03-29,"""N-MED""",618.0000,12794.78,"""Sexta-feira Santa""",1,0,"""Nenhuma""",0
1569,2024-06-12,"""N-MED""",1149.0000,24120.85,"""Dia Comum""",0,1,"""Dia dos Namorados""",0
1704,2024-11-22,"""N-MED""",1317.0000,39134.22,"""Dia Comum""",0,0,"""Nenhuma""",1
1572,2025-05-11,"""MED""",429.0000,14108.55,"""Dia Comum""",0,1,"""Dia das Mães""",0


In [ ]:
# df_final.sort("total_faturamento", descending=False).head(10)
# df_final.drop_nans().sort("total_faturamento", descending=True).head(10)

In [ ]:
# df_final.head(df_final['codigo_filial'] == "1623" and df_final['data_emissao'] == "2024-01-25")

In [ ]:
# df_grouped.head(df_final['codigo_filial'] == "1623" and df_final['data_emissao'] == "2024-01-25")